<a href="https://colab.research.google.com/github/KULDEEPSONI-source/MACHINE-LEARNING/blob/main/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dimensionality reduction is the process of reducing the number of input variables (features) in a dataset while retaining as much information as possible. It is a cornerstone technique in machine learning and data science, used to combat the "curse of dimensionality," improve model performance, and simplify data visualization.

---

### Why Use Dimensionality Reduction?

* **Computational Efficiency:** Fewer dimensions mean faster training times and lower memory usage for algorithms.
* **Noise Reduction:** By focusing on the "most important" patterns, you often discard irrelevant noise or redundant features.
* **Visualization:** It is impossible to visualize data beyond three dimensions; techniques like PCA or t-SNE allow us to project complex data into 2D or 3D space.
* **Improved Model Accuracy:** It helps prevent **overfitting** by simplifying the model and focusing on the most informative features.

---

### Key Techniques

Dimensionality reduction is generally categorized into two approaches: **Feature Selection** and **Feature Extraction**.

#### 1. Feature Selection

This involves choosing a subset of the original features. You do not transform the data; you simply drop the features that contribute the least to the predictive power of the model.

* **Filter Methods:** Use statistical measures (e.g., correlation coefficient) to score features.
* **Wrapper Methods:** Use a specific model to evaluate combinations of features.
* **Embedded Methods:** Perform feature selection during the model training process (e.g., LASSO regression).

#### 2. Feature Extraction (Linear & Non-Linear)

This transforms the data from a high-dimensional space to a lower-dimensional space. The resulting features are new, "latent" variables that represent the original data.

| Technique | Type | Best Used For |
| --- | --- | --- |
| **PCA (Principal Component Analysis)** | Linear | Reducing dimensionality while preserving the maximum variance of the data. |
| **LDA (Linear Discriminant Analysis)** | Linear | Supervised tasks where you need to maximize class separability. |
| **t-SNE** | Non-Linear | Visualizing high-dimensional datasets in 2D or 3D. |
| **UMAP** | Non-Linear | Similar to t-SNE but often faster and better at preserving global structure. |
| **Autoencoders** | Non-Linear | Deep learning-based compression where a neural network learns a compressed representation. |

---

### The Trade-offs

While powerful, dimensionality reduction is not without risks:

* **Information Loss:** By condensing data, you inherently lose some granularity. You must balance the number of dimensions kept against the amount of variance explained.
* **Interpretability:** If you use **Feature Extraction** (like PCA), the new features (Principal Components) are linear combinations of the original variables. This makes it difficult to explain to stakeholders exactly what "Component 1" represents compared to the original features.
* **Computational Overhead:** Techniques like t-SNE or complex Autoencoders can be computationally expensive to compute initially.

Would you like to explore how to implement one of these techniques—such as PCA—using a programming language like Python?

# ***Feature Elimination (RFE) and Sequential Feature Selection (SFS) are wrapper methods ***

-----
----
Both Recursive Feature Elimination (RFE) and Sequential Feature Selection (SFS) are **wrapper methods** for feature selection. They use a machine learning model to evaluate subsets of features to determine which ones provide the best predictive performance.

---

### 1. Recursive Feature Elimination (RFE)

RFE is a greedy optimization algorithm. It works by training a model, identifying the least important feature, removing it, and repeating the process until the desired number of features is reached.

#### Key Parameters (scikit-learn implementation):

* **`estimator`**: The machine learning model used for ranking features (must expose either `coef_` or `feature_importances_`).
* **`n_features_to_select`**: The number of features you want to keep. If `None`, half the features are removed.
* **`step`**: The number of features to remove at each iteration. A value of 1 removes one feature at a time; an integer > 1 removes multiple features simultaneously for speed.
* **`importance_getter`**: Specifies how to extract feature importance if the estimator is a pipeline or a custom object.

#### How it works:

1. Train the model on the full feature set.
2. Rank features based on their weights or importance scores.
3. Prune the weakest feature(s).
4. Repeat until the target number of features is reached.

---

### 2. Sequential Feature Selection (SFS)

SFS is a iterative approach that adds or removes features one by one based on a performance metric (e.g., cross-validation accuracy). Unlike RFE, it does not rely on model-specific internal weights (like `coef_`).

#### Key Parameters:

* **`estimator`**: The machine learning model used to evaluate feature subsets.
* **`n_features_to_select`**: The target number of features to select.
* **`direction`**:
* `'forward'`: Starts with zero features and adds the "best" one in each step.
* `'backward'`: Starts with all features and removes the "worst" one in each step.


* **`scoring`**: The performance metric used to evaluate the model (e.g., `accuracy`, `r2`, `neg_mean_squared_error`).
* **`cv`**: Cross-validation splitting strategy (e.g., 5-fold). This is crucial as it ensures the selected features generalize well.

#### How it works:

* **Forward:** Start empty. Add the feature that yields the highest score. Repeat until the target count is met.
* **Backward:** Start full. Remove the feature that, when absent, results in the highest score. Repeat until the target count is met.

---

### Comparison Table

| Feature | RFE | SFS |
| --- | --- | --- |
| **Dependency** | Requires model weights (`coef_` or `feature_importances_`). | Model-agnostic; works with any estimator. |
| **Logic** | Prunes based on internal model importance. | Prunes based on cross-validated performance metrics. |
| **Computational Cost** | Generally faster; one-pass importance ranking. | Slower; requires multiple model trainings per step (due to CV). |
| **Flexibility** | Limited to models with feature attributes. | Highly flexible; can optimize for specific metrics. |

---

### Which one to choose?

* Use **RFE** if your estimator provides feature importances (e.g., Linear Regression, Random Forest, SVM) and you have a very large dataset where training repeatedly is too expensive.
* Use **SFS** if you want to optimize for a specific metric (like F1-score or Log-loss) that your model doesn't natively optimize, or if your model does not provide built-in feature importance scores.

-----
----
--

# **FE-RFE**

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [19]:
df= pd.read_csv('//content/Customer Churn.csv')

In [20]:
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [22]:
df['TotalCharges']= pd.to_numeric(df['TotalCharges'],errors='coerce')

In [23]:
df['TotalCharges'].isnull().sum()

np.int64(11)

In [24]:
df.dropna(how='any',inplace=True)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [26]:
df.drop(['customerID'],axis='columns',inplace=True)

In [27]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [28]:
df=pd.get_dummies(df,drop_first=True)

In [29]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,True,False,True,False,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,True,False,False,False,False,True,False
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,True,False,False,True,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,True,False,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,True,False,True,False,True


In [30]:
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

In [31]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,1,29.85,29.85,0,1,0,0,1,0,...,0,0,0,0,0,1,0,1,0,0
1,0,34,56.95,1889.50,1,0,0,1,0,0,...,0,0,0,1,0,0,0,0,1,0
2,0,2,53.85,108.15,1,0,0,1,0,0,...,0,0,0,0,0,1,0,0,1,1
3,0,45,42.30,1840.75,1,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
4,0,2,70.70,151.65,0,0,0,1,0,0,...,0,0,0,0,0,1,0,1,0,1


In [32]:
df.Churn_Yes.value_counts()/len(df)*100

,count
Churn_Yes,
0,73.421502
1,26.578498


In [33]:
# Considering 'Churn' --> y-variable
# Other columns --> X-variable

X = df.drop(['Churn_Yes'], axis=1)
y = df['Churn_Yes']

In [34]:
X

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56.95,1889.50,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42.30,1840.75,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70.70,151.65,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,24,84.80,1990.50,1,1,1,1,0,1,...,0,1,0,1,1,0,1,0,0,1
7039,0,72,103.20,7362.90,0,1,1,1,0,1,...,0,1,0,1,1,0,1,1,0,0
7040,0,11,29.60,346.45,0,1,1,0,1,0,...,0,0,0,0,0,0,1,0,1,0
7041,1,4,74.40,306.60,1,1,0,1,0,1,...,0,0,0,0,0,0,1,0,0,1


In [35]:
y


,Churn_Yes
0,0
1,0
2,1
3,0
4,1
...,...
7038,0
7039,0
7040,0
7041,1


In [36]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

---
----

To fully understand how `RFE` works in `scikit-learn`, let's break down the parameters of the constructor: `RFE(estimator, *, n_features_to_select=None, step=1, verbose=0, importance_getter='auto')`.

### Parameter Breakdown

| Parameter | Type | Description |
| --- | --- | --- |
| **`estimator`** | Estimator Object | The supervised learning estimator (e.g., `LogisticRegression()`, `RandomForestClassifier()`) that provides information about feature importance through either a `coef_` or `feature_importances_` attribute. |
| **`n_features_to_select`** | `int` or `float` | The number of features you want to keep. If an `int`, it is the absolute number. If a `float` between 0 and 1, it represents the percentage of features to select. If `None`, half of the features are selected. |
| **`step`** | `int` or `float` | Determines how many features are removed at each iteration. If `int`, it removes that many features per step. If `float` (0.0 to 1.0), it removes a percentage of the remaining features per step. |
| **`verbose`** | `int` | Controls the output detail. Set to `1` (or higher) to see the progress of the feature removal process in your console. |
| **`importance_getter`** | `str` or `callable` | Defines how to extract importance. `'auto'` uses `coef_` first, then `feature_importances_`. You can pass a string naming the attribute or a callable to extract importance from a custom model. |

---

### Understanding the Mechanism

The "Recursive" part of RFE means that at every iteration, the model is trained, the least important features are removed, and the model is retrained on the smaller subset. This continues until the desired `n_features_to_select` is reached.

### Practical Tips

* **The `step` parameter:** Setting a `step` higher than 1 (e.g., `step=5`) will make the algorithm run much faster because it removes features in chunks rather than one by one. This is highly recommended for datasets with thousands of features.
* **Model Compatibility:** Ensure your `estimator` is compatible. Since you are using `LogisticRegression`, it works because the model exposes `coef_` after training. If you used a model that doesn't provide coefficients or feature importances, RFE would raise an error.
* **Performance:** Because RFE fits the model multiple times, it can be computationally expensive. Always check if your model supports `n_jobs` (parallel processing) or if a simpler method like `SelectFromModel` (which performs a single pass) might suffice.

----
----
----
---


In [37]:
# Apply Recursive Feature Elimination (RFE) to select 5 best features

model = LogisticRegression()
rfe = RFE(model, n_features_to_select=5)

In [38]:
rfe=rfe.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

----
---
----
---
To ensure you fully understand how to bridge the gap between "selected features" and "model training," let's break down the mechanics behind the code you just implemented.

Since you are applying the result of `rfe.support_` to your dataframes, you are essentially performing a **feature subset selection**. Here is the breakdown of the objects and properties involved:

### The Components of Your Code

* **`rfe.support_`**: This is an array of boolean values (True/False) with a length equal to the number of columns in your original `X_train`.
* `True` indicates the feature was kept (ranked #1).
* `False` indicates the feature was eliminated during the recursive process.


* **`X_train.columns[...]`**: This is a pandas indexing operation. By passing the boolean mask `rfe.support_` into the columns, you are filtering your list of feature names to keep only those that returned `True`.
* **`X_train[selected_features]`**: This creates a **new view** (a subset) of your dataframe. It effectively "drops" every column that was not deemed important enough by your `LogisticRegression` model.

---

### Why this is the "Golden Rule" of Feature Selection

The most important takeaway here is **Consistency**. When you use `rfe.support_` to subset your `X_test_selected`, you are ensuring that the model sees the exact same input features in the exact same column order as it did during training.

If you were to accidentally re-run RFE on the test set, or select features based on the test set's own variance, you would introduce **Data Leakage**, which leads to overly optimistic performance metrics that will fail in a real-world production environment.

### Summary Checklist

Before you proceed to fit your model on `X_train_selected`, ensure you have confirmed these three things:

1. **Feature Alignment**: Does `X_train_selected.columns.equals(X_test_selected.columns)` return `True`? (It should).
2. **Model Input**: Are you passing these new dataframes into a *fresh* instance of your model (i.e., a model not previously fitted with all features)?
3. **Scale**: If your `LogisticRegression` model requires scaling (e.g., `StandardScaler`), ensure you fit the scaler only on `X_train_selected` and then `.transform()` both the train and test subsets.
----
---
---
---


In [39]:
# Get the selected features

selected_features = X_train.columns[rfe.support_]
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

In [40]:
# Build the model

model_orig = LogisticRegression()
model_orig.fit(X_train, y_train)
y_pred_orig = model_orig.predict(X_test)

accuracy_orig = accuracy_score(y_test, y_pred_orig)
print("Accuracy of the base model is: ", round(accuracy_orig*100, 2))

Accuracy of the base model is:  79.24


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [41]:
# Build the RFE based model on 5 predictors

model_rfe = LogisticRegression()
model_rfe.fit(X_train_selected, y_train)
y_pred_rfe = model_rfe.predict(X_test_selected)

accuracy_rfe = accuracy_score(y_test, y_pred_rfe)
print("Accuracy of the RFE based model is: ", round(accuracy_rfe*100, 2))

Accuracy of the RFE based model is:  76.3


-----
----
end of REF

# **START OF SFS**

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [43]:
df = pd.read_csv('//content/Customer Churn.csv')

In [44]:
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [46]:
# Converting TotalCharges to numericals
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [47]:
df['TotalCharges'].isnull().sum()

np.int64(11)

In [48]:
# NaN Imputation

df.dropna(how='any', inplace=True)

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [50]:
df.drop(['customerID'], axis='columns', inplace=True)

In [51]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [52]:
# Dummy Encoding

df = pd.get_dummies(df, drop_first=True)

In [53]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,True,False,True,False,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,True,False,False,False,False,True,False
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,True,False,False,True,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,True,False,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,True,False,True,False,True


In [54]:
df.Churn_Yes.value_counts()/len(df)*100

,count
Churn_Yes,
False,73.421502
True,26.578498


In [55]:
# Considering 'Churn' --> y-variable
# Other columns --> X-variable

X = df.drop(['Churn_Yes'], axis=1)
y = df['Churn_Yes']

In [56]:
X

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,24,84.80,1990.50,True,True,True,True,False,True,...,False,True,False,True,True,False,True,False,False,True
7039,0,72,103.20,7362.90,False,True,True,True,False,True,...,False,True,False,True,True,False,True,True,False,False
7040,0,11,29.60,346.45,False,True,True,False,True,False,...,False,False,False,False,False,False,True,False,True,False
7041,1,4,74.40,306.60,True,True,False,True,False,True,...,False,False,False,False,False,False,True,False,False,True


In [57]:
y

,Churn_Yes
0,False
1,False
2,True
3,False
4,True
...,...
7038,False
7039,False
7040,False
7041,True


In [58]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [59]:
# Build the model

model_orig = LogisticRegression()
model_orig.fit(X_train, y_train)
y_pred_orig = model_orig.predict(X_test)

accuracy_orig = accuracy_score(y_test, y_pred_orig)
print("Accuracy of the base model is: ", round(accuracy_orig*100, 2))

Accuracy of the base model is:  79.24


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [60]:
# Apply Successive Feature Selection (SFS) to select 5 best features

model = LogisticRegression()
sfs = SequentialFeatureSelector(model, n_features_to_select=5)

In [61]:
# Fitting the model
sfs = sfs.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

In [62]:
len(sfs.feature_names_in_)

30

In [63]:
sfs.get_support()

array([False,  True, False, False, False, False, False, False,  True,
       False,  True, False, False, False, False, False, False, False,
       False, False, False, False, False,  True, False, False, False,
       False,  True, False])

In [64]:
# Get the selected features

selected_features = X.columns[sfs.get_support()]
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

In [65]:
# Build the RFE based model on 5 predictors

model_sfs = LogisticRegression()
model_sfs.fit(X_train_selected, y_train)
y_pred_sfs = model_sfs.predict(X_test_selected)

accuracy_sfs = accuracy_score(y_test, y_pred_sfs)
print("Accuracy of the RFE based model is: ", round(accuracy_sfs*100, 2))

Accuracy of the RFE based model is:  79.15


---
---
---
END OF SFS



The **Chi-Square ($\chi^2$) test** is a statistical method used in feature engineering to select the most relevant features for a **classification** model. It works by measuring the dependence between two categorical variables: your **input feature** and your **target variable**.

### How It Works

The core idea is to test the null hypothesis ($H_0$) that a feature and the target variable are **independent**.

1. **Contingency Table:** You build a table counting the occurrences of each category in your feature against each class in your target variable (Observed frequencies).
2. **Expected Frequencies:** You calculate what those counts *would* look like if the feature had absolutely no relationship with the target (i.e., if they were totally independent).
3. **$\chi^2$ Statistic:** You calculate the deviation between Observed ($O$) and Expected ($E$) frequencies:

$$\chi^{2} = \sum \frac{(O - E)^2}{E}$$


4. **Selection:** * A **high $\chi^2$ score** means the observed data deviates significantly from what we would expect if the variables were independent. This implies the feature **is dependent** on the target and likely carries useful predictive information.
* A **low $\chi^2$ score** suggests the feature and target are likely independent, making the feature a good candidate to drop.



---

### When to Use It

* **Target Type:** Must be categorical (e.g., Yes/No, Class A/B/C).
* **Feature Type:** Must be categorical. If your features are continuous, you must discretize (bin) them first or use a different method like ANOVA or Correlation.
* **Data Requirements:** `scikit-learn`'s implementation (`chi2`) requires your categorical data to be label-encoded (transformed into non-negative integers).

### Practical Implementation

In Python, you typically use `SelectKBest` from `sklearn` to automate the selection process.

```python
from sklearn.feature_selection import SelectKBest, chi2

# Assume X contains label-encoded categorical features and y is the target
# Select the top 5 features with the highest Chi-Square scores
selector = SelectKBest(score_func=chi2, k=5)
X_new = selector.fit_transform(X, y)

# You can view the scores for each feature
feature_scores = dict(zip(X.columns, selector.scores_))

```

### Limitations to Keep in Mind

* **Independence Assumption:** The test assesses features individually (univariate). It **does not** account for interactions between features (e.g., two features might be weak individually but powerful together).
* **Small Sample Sizes:** The $\chi^2$ test can be unreliable if your contingency table cells have very low frequencies (generally, if the expected value in any cell is less than 5).
* **Categorical Only:** It is strictly for categorical data; don't attempt to use it on raw continuous data without proper binning.

[Chi-Square Test for Feature Selection Explained](https://www.youtube.com/watch?v=aSghZSEe8qg)


In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [67]:
df = pd.read_csv('//content/Customer Churn.csv')

In [68]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [69]:
# Converting TotalCharges to numericals
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [70]:
# NaN Imputation

df.dropna(how='any', inplace=True)

In [71]:
df.drop(['customerID'], axis='columns', inplace=True)

In [72]:
# Dummy Encoding

df = pd.get_dummies(df, drop_first=True)

In [73]:
df.Churn_Yes.value_counts()/len(df)*100

,count
Churn_Yes,
False,73.421502
True,26.578498


In [74]:
# Considering 'Churn' --> y-variable
# Other columns --> X-variable

X = df.drop(['Churn_Yes'], axis=1)
y = df['Churn_Yes']

In [75]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [76]:
# Build the model

model_orig = LogisticRegression()
model_orig.fit(X_train, y_train)
y_pred_orig = model_orig.predict(X_test)

accuracy_orig = accuracy_score(y_test, y_pred_orig)
print("Accuracy of the base model is: ", round(accuracy_orig*100, 2))

Accuracy of the base model is:  79.24


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [77]:
# Apply chi2

chi2_selector = SelectKBest(chi2, k=5)
X_train_chi2 = chi2_selector.fit_transform(X_train, y_train)

X_test_chi2 = chi2_selector.transform(X_test)

Your implementation of `SelectKBest` with the `chi2` scoring function is correct and follows the standard machine learning workflow. By fitting on the training set and transforming both the train and test sets, you are properly handling the data and avoiding leakage.

### Understanding the Parameters

To ensure you have full control over this selector, here is a breakdown of the parameters you are using:

| Parameter | Description |
| --- | --- |
| **`score_func`** | The function used to calculate the relevance of features. By setting this to `chi2`, you are specifically targeting categorical dependencies. |
| **`k`** | The number of top-performing features to keep. Setting this to `5` will return the 5 features with the highest $\chi^2$ statistics. |

---

### Key Attributes for Inspection

After `fit_transform` is called, the `chi2_selector` object contains information you can use to validate your model:

* **`chi2_selector.scores_`**: An array containing the $\chi^2$ statistic for *every* input feature. A higher value indicates a stronger dependence between the feature and the target.
* **`chi2_selector.pvalues_`**: An array containing the p-values associated with each statistic. A very low p-value (typically $< 0.05$) suggests the relationship is statistically significant and not just due to random chance.

### Essential Checklist for Chi-Square

Before proceeding, ensure your data meets these requirements to avoid unexpected errors or invalid results:

1. **Non-Negative Values:** The $\chi^2$ test requires input data to be non-negative. If you have negative values (e.g., from `StandardScaler`), your code will throw an error. Use `MinMaxScaler` if you need to scale your data before applying `chi2`.
2. **Categorical Data:** `chi2` is specifically designed for categorical features. If you are feeding continuous data directly into it, the statistical assumptions of the test are violated.
3. **Consistency:** Just like with RFE, you have used the correct logic by calling `.fit_transform()` on `X_train` and only `.transform()` on `X_test`. This ensures the specific features selected during the training phase are the only ones used during the testing phase.


In [78]:
# Build the model

model_chi2 = LogisticRegression()
model_chi2.fit(X_train_chi2, y_train)
y_pred_chi2 = model_chi2.predict(X_test_chi2)

accuracy_chi2 = accuracy_score(y_test, y_pred_chi2)
print("Accuracy of the chi2 based model is: ", round(accuracy_chi2*100, 2))

Accuracy of the chi2 based model is:  78.2


You have successfully completed the pipeline for building a model using features selected via the Chi-Square test. Your implementation is clean and follows standard practices.

### What you have achieved

1. **Feature Selection:** You used the $\chi^2$ statistic to identify the 5 features with the strongest statistical dependence on your target variable.
2. **Model Fitting:** You trained your `LogisticRegression` model exclusively on the reduced feature space.
3. **Evaluation:** You quantified the performance of this specific subset, allowing you to see how the model behaves when it only focuses on the most "informative" features according to the statistical test.

---

### Understanding the Pipeline

To visualize how your data flows through this process, consider this progression:

### Next Steps for Deeper Insight

Since you now have an accuracy score, you might want to compare this against the full model or other selection methods to see if your feature engineering actually improved performance. Here is how you can perform a quick comparison:

* **Compare with "Baseline":** Train the same `LogisticRegression` model on the **original** `X_train` (before `SelectKBest`) and compare the test accuracy.
* **Evaluate Feature Importance:** If your model is a `LogisticRegression`, you can inspect the coefficients (`model_chi2.coef_`) to see the magnitude and direction of the impact each of the 5 selected features has on the prediction.
* **Check for Overfitting:** If the training accuracy is significantly higher than your `accuracy_chi2` on the test set, your model might be overfitting, even with only 5 features. You might consider adding regularization (e.g., `LogisticRegression(penalty='l2', C=0.1)`).

### A Note on Metrics

While `accuracy_score` is a great starting point, if your target classes are imbalanced (e.g., 90% class 0 and 10% class 1), accuracy can be misleading. You may want to look at a **Confusion Matrix** or **Classification Report** to see how the model is performing on each class individually:

```python
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_chi2))
print(classification_report(y_test, y_pred_chi2))

```



# **CHI SQUARE END---**
----


---
---
---
Here's a crisp guide to **PCA in Feature Engineering**, along with the best YouTube resources to go deeper.

---

## 🔷 What is PCA?

PCA (Principal Component Analysis) is a statistical procedure that converts a set of **correlated variables** into a set of **uncorrelated variables**. It reduces the number of features in a dataset while keeping the most important information, transforming correlated features into a smaller set of uncorrelated components called **principal components**.

---

## 🔷 Why Use PCA in Feature Engineering?

The reason to apply PCA is that the information in the predictors could be more efficiently stored — in other words, there is **redundancy in the data**.

Dimensionality reduction techniques like PCA can improve **computational efficiency** and reduce **overfitting** by capturing the most important aspects of high-dimensional data.

---

## 🔷 Two Ways PCA is Used in Feature Engineering

**1. As a descriptive technique** — since the components capture variation, you can compute Mutual Information (MI) scores for the components to see what kind of variation is most predictive of the target variable. This gives ideas for new features to create.

**2. As a direct transformation** — replacing original features with principal components to feed into a model, reducing dimensionality while retaining variance.

---

## 🔷 Key Intuition

PCA finds vectors (eigenvectors) that define some **hidden feature** inherent in the data. The math has no idea what "bigness" or "size" means, but it finds the vector that corresponds to what we'd call an important underlying property — making PCA a powerful **feature extraction tool**.

Intuitively, PCA can supply the user with a lower-dimensional picture — a **projection or "shadow"** of the data when viewed from its most informative viewpoint.

---

## 🔷 Important Note

The dataset on which PCA is applied **must be scaled** first. Results are sensitive to the relative scaling of features.

---

## 📺 Best YouTube Videos

| # | Video | Channel | Best For |
|---|-------|---------|----------|
| 1 | https://youtu.be/iRbsBi5W0-c?si=pae7BEl-JOdPB9RE CAMPUSX PART1

**StatQuest** is widely considered the gold standard for understanding PCA intuitively — highly recommended as your first stop.
---
---
---


In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [80]:
df = pd.read_csv('//content/Customer Churn.csv')

In [81]:
df.head(5)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [82]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [83]:
# Converting TotalCharges to numericals
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [84]:
df['TotalCharges'].isnull().sum()

np.int64(11)

In [85]:
# NaN Imputation

df.dropna(how='any', inplace=True)

In [86]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [87]:
df.drop(['customerID'], axis='columns', inplace=True)

In [88]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [89]:
# Dummy Encoding

df = pd.get_dummies(df, drop_first=True)

In [90]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,True,False,True,False,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,True,False,False,False,False,True,False
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,True,False,False,True,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,True,False,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,True,False,True,False,True


In [91]:
df.Churn_Yes.value_counts()/len(df)*100

,count
Churn_Yes,
False,73.421502
True,26.578498


In [92]:
# Considering 'Churn' --> y-variable
# Other columns --> X-variable

X = df.drop(['Churn_Yes'], axis=1)
y = df['Churn_Yes']

In [93]:
X

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,24,84.80,1990.50,True,True,True,True,False,True,...,False,True,False,True,True,False,True,False,False,True
7039,0,72,103.20,7362.90,False,True,True,True,False,True,...,False,True,False,True,True,False,True,True,False,False
7040,0,11,29.60,346.45,False,True,True,False,True,False,...,False,False,False,False,False,False,True,False,True,False
7041,1,4,74.40,306.60,True,True,False,True,False,True,...,False,False,False,False,False,False,True,False,False,True


In [94]:
y

,Churn_Yes
0,False
1,False
2,True
3,False
4,True
...,...
7038,False
7039,False
7040,False
7041,True


In [95]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [96]:
# Build the model

model_orig = LogisticRegression()
model_orig.fit(X_train, y_train)
y_pred_orig = model_orig.predict(X_test)

accuracy_orig = accuracy_score(y_test, y_pred_orig)
print("Accuracy of the base model is: ", round(accuracy_orig*100, 2))

Accuracy of the base model is:  79.24


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Applying PCA

In [97]:
from sklearn.decomposition import PCA

pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

In [98]:
explained_variance = pca.explained_variance_ratio_

print(explained_variance)

[9.99858069e-01 1.24090191e-04 1.71403599e-05 1.23908957e-07
 6.46782051e-08 5.36963154e-08 4.90502886e-08 4.47010962e-08
 4.19454064e-08 4.04476168e-08]


In [99]:
# Training the logistic regression model on training set

model_pca = LogisticRegression()
model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)

accuracy_pca = accuracy_score(y_test, y_pred_pca)
print("Accuracy of the base model is: ", round(accuracy_pca*100, 2))

Accuracy of the base model is:  79.57


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# **end of pca**

---
----
---**Linear Discriminant Analysis (LDA)** is a dimensionality reduction technique similar to PCA, but with one critical difference: **LDA is supervised.**

While PCA tries to find the axes that maximize the *total variance* of the data (ignoring class labels), LDA finds the axes that maximize the **separability** between known classes.

### How It Works

LDA aims to project your data into a lower-dimensional space while achieving two goals simultaneously:

1. **Maximize the distance** between the means of different classes.
2. **Minimize the spread (variance)** within each individual class.

### When to Use LDA

* **Classification Tasks:** Because LDA uses class labels, it is specifically designed to make classification models work better.
* **Multiclass Problems:** LDA can reduce the number of features to at most $C-1$ (where $C$ is the number of classes).
* **Supervised Learning:** Use LDA when you have labeled data and your primary goal is to improve the performance of a classifier, rather than just compressing data.

### Implementation in Python

```python
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# n_components is limited to (number of classes - 1)
lda = LDA(n_components=1)
X_train_lda = lda.fit_transform(X_train, y_train)

X_test_lda = lda.transform(X_test)

```

### Key Differences: PCA vs. LDA

| Feature | PCA | LDA |
| --- | --- | --- |
| **Type** | Unsupervised | Supervised |
| **Goal** | Maximize variance | Maximize class separability |
| **Labels** | Does not use labels | Uses labels (`y_train`) |
| **Output** | Any number of components | Max $C-1$ components |

### Learning Resources
https://youtu.be/9LO-bj1jyz4?si=7Zjf9ygIVgzy1UYM

* **[StatQuest: Linear Discriminant Analysis](https://www.youtube.com/watch?v=azXCzI57Yfc)**: A clear, conceptual guide on why LDA is a "supervised" approach and how it finds the optimal projection for classification.
* **[Sebastian Raschka's Tutorial](https://sebastianraschka.com/Articles/2014_python_lda.html)**: An excellent technical deep-dive into the math and application of LDA for feature extraction.

### Important Considerations

* **Normality Assumption:** LDA assumes that your data is normally distributed and that classes have the same covariance matrix. If these assumptions are heavily violated, LDA may perform poorly.
* **Overfitting:** Because LDA uses class labels, it can overfit if you have very few samples compared to the number of features.
----
----
---
No, in the context of Linear Discriminant Analysis (LDA), **"class" and "features" are two completely different things.**

To clarify:

* **Features:** These are your input variables (the columns in your data, like 'Age', 'Income', 'Temperature', etc.). These are the variables you are using to make a prediction.
* **Class:** This is your **target variable** (the label you are trying to predict). For example, if you are predicting "Will a customer buy a product?", the classes are "Yes" and "No".

### The Relationship in LDA

LDA is **supervised**, which means it needs to know the "class" (the answer) to find the best way to separate your "features."

* **The Goal:** LDA looks at your **features** and tries to find a new mathematical combination of them that makes the **classes** as distinct as possible.
* **The Limitation:** Because LDA uses these class labels to calculate the separation, you are limited by how many classes you have. You can only reduce your data to a maximum of **(Number of Classes - 1)** components.
* If you have 2 classes (Yes/No), LDA can only reduce your features down to 1 dimension.
* If you have 3 classes (Low/Medium/High), LDA can reduce your features down to at most 2 dimensions.



### Why this distinction matters

When you write your code:

```python
# X_train is your features
# y_train is your class (target)
lda.fit(X_train, y_train)

```

You are explicitly telling the algorithm: *"Use these **features** to create a separation between these specific **classes**."*

If you were doing **PCA**, you wouldn't use `y_train` at all because PCA doesn't care about the classes—it only looks at the variance within the features.


In [100]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [101]:
df = pd.read_csv('//content/Customer Churn.csv')

In [102]:
df.head(5)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [103]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [104]:
# Converting TotalCharges to numericals
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [105]:
df['TotalCharges'].isnull().sum()

np.int64(11)

In [106]:
# NaN Imputation

df.dropna(how='any', inplace=True)

In [107]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [108]:
df.drop(['customerID'], axis='columns', inplace=True)

In [109]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [110]:
# Dummy Encoding

df = pd.get_dummies(df, drop_first=True)

In [111]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,True,False,True,False,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,True,False,False,False,False,True,False
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,True,False,False,True,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,True,False,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,True,False,True,False,True


In [112]:
df.Churn_Yes.value_counts()/len(df)*100

,count
Churn_Yes,
False,73.421502
True,26.578498


In [113]:
# Considering 'Churn' --> y-variable
# Other columns --> X-variable

X = df.drop(['Churn_Yes'], axis=1)
y = df['Churn_Yes']

In [114]:
X

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,24,84.80,1990.50,True,True,True,True,False,True,...,False,True,False,True,True,False,True,False,False,True
7039,0,72,103.20,7362.90,False,True,True,True,False,True,...,False,True,False,True,True,False,True,True,False,False
7040,0,11,29.60,346.45,False,True,True,False,True,False,...,False,False,False,False,False,False,True,False,True,False
7041,1,4,74.40,306.60,True,True,False,True,False,True,...,False,False,False,False,False,False,True,False,False,True


In [115]:
y

,Churn_Yes
0,False
1,False
2,True
3,False
4,True
...,...
7038,False
7039,False
7040,False
7041,True


In [116]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [117]:
# Build the model

model_orig = LogisticRegression()
model_orig.fit(X_train, y_train)
y_pred_orig = model_orig.predict(X_test)

accuracy_orig = accuracy_score(y_test, y_pred_orig)
print("Accuracy of the base model is: ", round(accuracy_orig*100, 2))

Accuracy of the base model is:  79.24


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# **Applying PCA**

In [118]:
from sklearn.decomposition import PCA

pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

In [119]:
explained_variance = pca.explained_variance_ratio_

print(explained_variance)

[9.99858069e-01 1.24090191e-04 1.71403599e-05 1.23908957e-07
 6.46782051e-08 5.36963154e-08 4.90502886e-08 4.47010962e-08
 4.19454064e-08 4.04476168e-08]


In [120]:
# Training the logistic regression model on training set

model_pca = LogisticRegression()
model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)

accuracy_pca = accuracy_score(y_test, y_pred_pca)
print("Accuracy of the PCA model is: ", round(accuracy_pca*100, 2))

Accuracy of the PCA model is:  79.57


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# **Applying** LDA

In [121]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis()
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

In [122]:
explained_variance = lda.explained_variance_ratio_

print(explained_variance)

[1.]


In [123]:
# Training the logistic regression model on training set

model_lda = LogisticRegression()
model_lda.fit(X_train_lda, y_train)
y_pred_lda = model_lda.predict(X_test_lda)

accuracy_lda = accuracy_score(y_test, y_pred_lda)
print("Accuracy of the LDA model is: ", round(accuracy_lda*100, 2))

Accuracy of the LDA model is:  79.53


# **END OF LDA**

In feature engineering, **KPCA (Kernel Principal Component Analysis)** and **QDA (Quadratic Discriminant Analysis)** serve very different purposes. One is a non-linear dimensionality reduction technique, while the other is a classification algorithm that can be adapted for feature selection.

---

### 1. KPCA (Kernel Principal Component Analysis)

Standard PCA is linear—it can only find straight lines to summarize your data. **KPCA** extends this by using the **"Kernel Trick,"** allowing it to project data into a higher-dimensional space where non-linear patterns become linearly separable.

* **When to use it:** When your data has complex, non-linear relationships that a standard PCA would miss.
* **How it works:** It uses kernels (like RBF, Polynomial, or Sigmoid) to map input features into a high-dimensional feature space, then performs PCA in that space.
* **Trade-off:** It is computationally expensive for large datasets compared to standard PCA, and it is harder to interpret because you are performing operations in an abstract high-dimensional space.

### 2. QDA (Quadratic Discriminant Analysis)

While **LDA** assumes that different classes share the same covariance matrix (creating straight-line boundaries), **QDA** is more flexible. It assumes each class has its own unique covariance matrix.

* **Role in Feature Engineering:** QDA is technically a classifier, not a dimension reducer. However, it is used in feature engineering because its decision boundaries are **quadratic (curved)**, not linear.
* **The "Feature" Connection:** You can use the logic of QDA to perform **Feature Selection** by observing which features contribute most to the variance within the curved decision boundaries of each class.
* **When to use it:** When your classes are not linearly separable but have a quadratic relationship in the feature space. It is more powerful than LDA but prone to overfitting if you have limited data, as it must estimate more parameters.

---

### Comparison Summary

| Feature | KPCA | QDA |
| --- | --- | --- |
| **Primary Use** | Non-linear Dimensionality Reduction | Classification (Non-linear boundaries) |
| **Supervised?** | No (Unsupervised) | Yes (Supervised) |
| **Relationship** | Maps data to higher dimensions | Models class-specific distributions |
| **Computation** | High (Kernel matrix calculation) | Moderate (Matrix inversion) |

---

### Which one should you choose?

* If your goal is **to reduce the number of features** because your data is highly non-linear, use **KPCA**.
* If your goal is **to perform classification** where the separation between classes is curved rather than a straight line, use **QDA** as your model.

**Pro-Tip:** Because KPCA creates entirely new "synthetic" features based on kernels, it is best used when you are building a pipeline for complex datasets where feature interpretability is less important than predictive power.


https://youtu.be/ispBXqC34Ak?si=E6sNyZWbIS98FUqR

In [124]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [125]:
df = pd.read_csv('//content/Customer Churn.csv')

In [126]:
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [127]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [128]:
# Converting TotalCharges to numericals
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [129]:
df['TotalCharges'].isnull().sum()

np.int64(11)

In [130]:
# NaN Imputation

df.dropna(how='any', inplace=True)

In [131]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [132]:
df.drop(['customerID'], axis='columns', inplace=True)

In [133]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [134]:
# Dummy Encoding

df = pd.get_dummies(df, drop_first=True)

In [135]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,True,False,True,False,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,True,False,False,False,False,True,False
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,True,False,False,True,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,True,False,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,True,False,True,False,True


In [136]:
df.Churn_Yes.value_counts()/len(df)*100

,count
Churn_Yes,
False,73.421502
True,26.578498


In [137]:
# Considering 'Churn' --> y-variable
# Other columns --> X-variable

X = df.drop(['Churn_Yes'], axis=1)
y = df['Churn_Yes']

In [138]:
X

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,24,84.80,1990.50,True,True,True,True,False,True,...,False,True,False,True,True,False,True,False,False,True
7039,0,72,103.20,7362.90,False,True,True,True,False,True,...,False,True,False,True,True,False,True,True,False,False
7040,0,11,29.60,346.45,False,True,True,False,True,False,...,False,False,False,False,False,False,True,False,True,False
7041,1,4,74.40,306.60,True,True,False,True,False,True,...,False,False,False,False,False,False,True,False,False,True


In [139]:
y

,Churn_Yes
0,False
1,False
2,True
3,False
4,True
...,...
7038,False
7039,False
7040,False
7041,True


In [140]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [141]:
# Build the model

model_orig = LogisticRegression()
model_orig.fit(X_train, y_train)
y_pred_orig = model_orig.predict(X_test)

accuracy_orig = accuracy_score(y_test, y_pred_orig)
print("Accuracy of the base model is: ", round(accuracy_orig*100, 2))

Accuracy of the base model is:  79.24


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# **Applying PCA**

In [142]:
from sklearn.decomposition import PCA

pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

In [143]:
explained_variance = pca.explained_variance_ratio_

print(explained_variance)

[9.99858069e-01 1.24090191e-04 1.71403599e-05 1.23908957e-07
 6.46782051e-08 5.36963154e-08 4.90502886e-08 4.47010962e-08
 4.19454064e-08 4.04476168e-08]


In [144]:
# Training the logistic regression model on training set

model_pca = LogisticRegression()
model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)

accuracy_pca = accuracy_score(y_test, y_pred_pca)
print("Accuracy of the PCA model is: ", round(accuracy_pca*100, 2))

Accuracy of the PCA model is:  79.57


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# **Applying LDA**

In [145]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis()
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

In [146]:
explained_variance = lda.explained_variance_ratio_

print(explained_variance)

[1.]


In [147]:
# Training the logistic regression model on training set

model_lda = LogisticRegression()
model_lda.fit(X_train_lda, y_train)
y_pred_lda = model_lda.predict(X_test_lda)

accuracy_lda = accuracy_score(y_test, y_pred_lda)
print("Accuracy of the LDA model is: ", round(accuracy_lda*100, 2))

Accuracy of the LDA model is:  79.53


# **Applying KernelPCA**

In [148]:
from sklearn.decomposition import KernelPCA

kpca = PCA(n_components=10)
X_train_kpca = kpca.fit_transform(X_train)
X_test_kpca = kpca.transform(X_test)

In [149]:
# Training the logistic regression model on training set

model_kpca = LogisticRegression()
model_kpca.fit(X_train_kpca, y_train)
y_pred_kpca = model_kpca.predict(X_test_kpca)

accuracy_kpca = accuracy_score(y_test, y_pred_kpca)
print("Accuracy of the PCA model is: ", round(accuracy_kpca*100, 2))

Accuracy of the PCA model is:  79.57


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


#Applying **QDA**

In [150]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

qda = QuadraticDiscriminantAnalysis()
X_train_qda = qda.fit(X_train, y_train)

y_pred_qda = qda.predict(X_test)

accuracy_qda = accuracy_score(y_test, y_pred_qda)
print("Accuracy of the PCA model is: ", round(accuracy_qda*100, 2))

Accuracy of the PCA model is:  70.28


/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
